In [1]:
import json
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
fs = sorted(Path("../data/json2").glob("*.json"))
print(f"{len(fs)} threads")

10257 threads


In [3]:
from IPython.display import display
# load output from
df3 = pd.read_parquet("../outputs/df_links4.parquet")
df3

,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url
title,,,,,,,,
Worth the Candle,1453,110,110,[{'author_flair_text': 'Self-Appointed Court S...,[https://reddit.com/r/rational/comments/f1rj3l...,2017-07-28 13:17:36,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...
"Alexander Wales - The Metropolitan Man, Shadows of the Limelight",1408,22,22,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/8mqrd0...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales]
A Practical Guide to Evil,1034,94,94,[{'author_flair_text': 'Ankh-Morpork City Watc...,[https://reddit.com/r/rational/comments/dutkzh...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ..."
Worm,925,83,83,[{'author_flair_text': 'Emergency Mustelid Hol...,[https://reddit.com/r/rational/comments/f1rj3l...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa..."
Mother of Learning,880,93,93,"[{'author_flair_text': 'Utopian Smut Peddler',...",[https://reddit.com/r/rational/comments/dutkzh...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...
...,...,...,...,...,...,...,...,...
Heartbreaker,-6,1,1,[{'author_flair_text': 'https://i.imgur.com/OQ...,[https://reddit.com/r/rational/comments/48akta...,2016-02-29 20:57:08,2016-02-29 20:57:08,[https://parahumans.wordpress.com/2012/01/17/b...
picking RPG clothes based on maxing stats instead of whether they match or not,-6,2,2,"[{'author_flair_text': None, 'body': 'It seems...",[https://reddit.com/r/rational/comments/9h1454...,2018-09-19 02:26:27,2018-09-19 04:51:09,[https://youtu.be/xq1tN9jZI80]
Orthogonality thesis,-8,2,2,"[{'author_flair_text': None, 'body': 'I don't ...",[https://reddit.com/r/rational/comments/3vc0si...,2015-12-04 18:17:07,2016-07-11 16:27:30,[https://wiki.lesswrong.com/wiki/Orthogonality...


## Extra get a llm summary of each link [WIP]

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [4]:
import dotenv
import os
from anycache import anycache

dotenv.load_dotenv()
from openai import OpenAI

# client = OpenAI()
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ['OPENROUTER_API_KEY'],
)

import tiktoken

# chose a model, compare price to the RAG MTED leaderboard https://huggingface.co/spaces/mteb/leaderboard
# and reward bench https://huggingface.co/spaces/allenai/reward-bench
# available models and prices https://openrouter.ai/models?context=64000&fmt=table&order=top-weekly&supported_parameters=structured_outputs&max_price=1
# note structured_outputs makes the price 3x

# GOOD
MODEL_NAME = "gpt-4o-mini" # ?b
cost = 0.150 / 1e6

# parsing faiil
# MODEL_NAME = "cohere/command-r-08-2024" # 32B

# no json on openrouter
# MODEL_NAME = "nousresearch/hermes-3-llama-3.1-70b"

# parsing faiil
MODEL_NAME = "meta-llama/llama-3.3-70b-instruct" # 70b

# json fail
MODEL_NAME = "meta-llama/llama-3.2-3b-instruct" # 3b

# # this works, by not use of optional or le or ge. a but incoherent
# MODEL_NAME = "cohere/command-r7b-12-2024" # 7b fail at tool calling

# this works, a bit incoherent
# MODEL_NAME = "nousresearch/hermes-2-pro-llama-3-8b"

# fails at json
# MODEL_NAME = "google/gemini-flash-1.5-8b"
# cost = 0.03 / 1e6

# GOOD!
MODEL_NAME = "google/gemini-flash-1.5"
cost = 0.075 / 1e6


enc = tiktoken.encoding_for_model('gpt-4')

In [5]:
# load all md posts 
md_posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in tqdm(fs):
    s = f.open().read()
    md_posts.append(s)


# order by date
def md2date(s: str) -> str:
    return s.split('* Created: ')[1].split('\n')[0]

md_posts = sorted(md_posts, key=md2date)
md2date(md_posts[0]), md2date(md_posts[-1])

  0%|          | 0/10263 [00:00<?, ?it/s]

('2009-11-25T02:34:03', '2024-12-23T15:00:14')

In [6]:

def get_post_context(urls: List[str], char_budget=100000, min_size=1000) -> str:
    # TODO maybe I should just get comment with links, and children?
    assert len(urls) > 0
    # from comments
    # return df3.loc[title].comments

    # or I could just all markdowns with

    # TODO use langchain chunking?

    matches = []
    for ii, post in enumerate(md_posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = char_budget / len(matches)
    budget_pp = max(budget_pp, min_size)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk

    # if too large get first N//2 and last N//2
    if len(s) > char_budget:
        s = s[:char_budget // 2] + "..." + s[-char_budget // 2:]
    return s


# url = df3.url[0].split('\n')
# print(url)
# c = get_context(url, 400000)
# print(c[:1000])

In [7]:
df3.iloc[-1].url

array(['https://sciencetrends.com/is-there-a-replicability-crisis-in-psychology-new-study-says-its-complicated/'],
      dtype=object)

In [8]:
# QC test with long and short context
# urls = ['https://archiveofourown.org/works/11478249/chapters/25740126',
#        'https://www.royalroad.com/fiction/25137/worth-the-candle',
#        'http://archiveofourown.org/works/11478249?view_full_work=true',
#        'http://archiveofourown.org/works/11478249/chapters/25740126',
#        'https://archiveofourown.org/works/11478249']
# c = get_post_context(urls, 400000)
# print(urls)
# print(c)

# # urls = ['https://www.amazon.com/Becoming-Batman-Possibility-Paul-Zehr/dp/0801890632']
# # c = get_post_context(urls, 400000)
# # print(urls)
# # print(c)


In [9]:


from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str = Field(description="Title of the fiction")
    description: str = Field(description="A few paragraphs of very concise, informative, dense, description of the fiction")
    tags: List[str] = Field(
        description="""Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: str = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote excerpts from every single readers' comments about the fiction"
    )
    reviews_summary: str = Field(
        description="Structured, dry, and concise summary of reviews"
    )
    reccomendations: str = Field(
        description="Why readers recommend the fiction"
    )
    disrecommendations: str = Field(
        description="Why readers disrecommend the fiction"
    )
    why: str = Field(
        description="Why/when might readers of r/rational like the fiction"
    )
    if_you_liked_x_you_will_like_this: List[str] = Field(
        description="Fans of X will also like the fiction. List all examples that readers mention wrt to the fiction."
    )

    rating_quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment of readers towards the fiction (out of 10), it's important to be consistent and use the same scale for all fictions"
    )
    rating_rationality: float = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: float
    rating_plot: float
    rating_character: float
    rating_worldbuilding: float

In [10]:
from openai.lib._pydantic import to_strict_json_schema

schema = to_strict_json_schema(FictionInfo)
schema = json.dumps(schema)
# print(schema)

In [11]:

def get_llm_summary(title: str, urls: str, context: str):
    chat_completion = client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": f"You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community recommendations from r/rational into a dry, informative, concise, and structured format for your own personal notes. Because it's private you can be consise, frank, and opinionated. Ignoring any authors promotion. You  answer in JSON. Here's the json schema you must adhere to:\n<schema>\n{schema}\n",
            },
            {
                "role": "user",
                "content": f"""Using the given structure, summarize the parts of the discussion which talk about the fiction: {title} (urls: {urls}).

### Discussion:

{context}

### Instructions

You are Gwern Branwern, using the given structure, summarize the above discussion of the fiction {title} (urls: {urls}).""",
            },
        ],
        model=MODEL_NAME,
        response_format=FictionInfo,
    )

    tokens = chat_completion.usage.prompt_tokens
    print(f"cost: {tokens * cost:.2f} USD, tokens: {tokens}")
    return tokens, chat_completion.choices[0].message.parsed.__dict__

In [12]:
# import shutil
# shutil.rmtree(f_cache_llm, ignore_errors=True)

In [13]:
output_dir = Path('../outputs/llm')
output_dir.mkdir(exist_ok=True)

In [14]:
# import shutil
# shutil.rmtree(output_dir, ignore_errors=True)

In [ ]:
llm_info = []

# TODO cache results as json
n_input_tokens = []

first = True

# l = len(df3)
# l = min(1000, len(df3))
for i in tqdm(range(len(df3))):
    title = df3.index[i]
    urls = df3.iloc[i].url

    fs_title = re.sub(r'[^\w\s]', '', title)
    f = output_dir / f"{i}_{fs_title}.json"

    context = get_post_context(urls, char_budget=120000)
    tokens = len(enc.encode(context))
    n_input_tokens.append(tokens)
    if f.exists():
        with f.open("r") as f:
            llm_data = json.load(f)
    else:            
        f_tokens, llm_data = get_llm_summary(title, urls, context)
        print(
            f"Input Tokens: {f_tokens}. Input Cost: {cost * f_tokens:.6f} USD, for title=`{title}`"
        )
        with f.open("w") as f:
            json.dump(llm_data, f)

        if first:
            c = f_tokens * len(df3) * cost / 2 # longest first, so assuming they get shorter and shorter
            print(f"Projected cost for all threads: {c:.5f} USD")
            print(f"Content: {context[:1000]}...")
            display(llm_data)
            first = False
    
    llm_data["title2"] = title
    llm_data["url"] = urls


    llm_info.append(llm_data)

  0%|          | 0/15209 [00:00<?, ?it/s]

In [ ]:
df_llm = pd.DataFrame(llm_info)
df_llm
# also join with df4

# then display as table